# Diffusion Process: Schedules, Forward Noising, and Sampling

Explore the DDPM mathematics: noise schedules, forward process, and reverse sampling.

In [ ]:
import sys

sys.path.insert(0, "src")

import torch
import numpy as np
import matplotlib.pyplot as plt

from from_zero.diffusion import (
    make_schedule,
    q_sample,
    predict_x0_from_noise,
    posterior_mean,
    sample_ddpm,
    make_synthetic_images,
    UNet,
    UNetConfig,
    load_diffusion_checkpoint,
)

## 1. Noise Schedules

In [ ]:
# Linear schedule
sched_linear = make_schedule("linear", timesteps=1000, beta_start=1e-4, beta_end=0.02)

# Cosine schedule
sched_cosine = make_schedule("cosine", timesteps=1000, cosine_s=0.008)

print(f"Linear: β₁={sched_linear.betas[0]:.6f}, βₜ={sched_linear.betas[-1]:.4f}")
print(f"Linear: ᾱ₁={sched_linear.alpha_bars[0]:.6f}, ᾱₜ={sched_linear.alpha_bars[-1]:.6f}")
print(f"Cosine: β₁={sched_cosine.betas[0]:.6f}, βₜ={sched_cosine.betas[-1]:.4f}")
print(f"Cosine: ᾱ₁={sched_cosine.alpha_bars[0]:.6f}, ᾱₜ={sched_cosine.alpha_bars[-1]:.6f}")

In [ ]:
# Plot schedules
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

t = torch.arange(1000)

axes[0, 0].plot(t, sched_linear.betas, label="Linear")
axes[0, 0].plot(t, sched_cosine.betas, label="Cosine")
axes[0, 0].set_title("βₜ (variance schedule)")
axes[0, 0].set_xlabel("Timestep t")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(t, sched_linear.alphas, label="Linear")
axes[0, 1].plot(t, sched_cosine.alphas, label="Cosine")
axes[0, 1].set_title("αₜ = 1 - βₜ (signal retained)")
axes[0, 1].set_xlabel("Timestep t")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(t, sched_linear.alpha_bars, label="Linear")
axes[1, 0].plot(t, sched_cosine.alpha_bars, label="Cosine")
axes[1, 0].set_title("ᾱₜ = ∏ αₛ (cumulative signal)")
axes[1, 0].set_xlabel("Timestep t")
axes[1, 0].set_yscale("log")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(t, sched_linear.posterior_variance, label="Linear")
axes[1, 1].plot(t, sched_cosine.posterior_variance, label="Cosine")
axes[1, 1].set_title("Posterior variance β̃ₜ")
axes[1, 1].set_xlabel("Timestep t")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Forward Process Visualization

In [ ]:
# Create a clean synthetic image
img = make_synthetic_images(1, image_size=32, seed=42)  # (1, 1, 32, 32)
x0 = img[0, 0]  # (32, 32)

# Sample at different timesteps
schedule = make_schedule("linear", timesteps=1000)
timesteps = [0, 100, 250, 500, 750, 999]

fig, axes = plt.subplots(1, len(timesteps), figsize=(15, 3))

for ax, t in zip(axes, timesteps):
    t_tensor = torch.tensor([t])
    noise = torch.randn_like(img)
    xt = q_sample(img, t_tensor, noise, schedule)

    ax.imshow(xt[0, 0], cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"t = {t}")
    ax.axis("off")

    # Show signal-to-noise ratio
    alpha_bar = schedule.alpha_bars[t].item()
    snr = alpha_bar / (1 - alpha_bar)
    ax.text(0.5, -0.15, f"SNR = {snr:.2f}", transform=ax.transAxes, ha="center", fontsize=8)

plt.suptitle("Forward Process: Clean → Noisy")
plt.tight_layout()
plt.show()

## 3. x₀ Recovery from Noise

In [ ]:
# Verify x₀ recovery identity
x0 = make_synthetic_images(1, image_size=32, seed=123)
noise = torch.randn_like(x0)
t = torch.tensor([500])
schedule = make_schedule("cosine", timesteps=1000)

xt = q_sample(x0, t, noise, schedule)
x0_recovered = predict_x0_from_noise(xt, t, noise, schedule)

mse = torch.mean((x0 - x0_recovered) ** 2).item()
max_err = torch.max(torch.abs(x0 - x0_recovered)).item()

print(f"MSE: {mse:.6f}")
print(f"Max abs error: {max_err:.6f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(x0[0, 0], cmap="gray", vmin=-1, vmax=1)
axes[0].set_title("Original x₀")
axes[1].imshow(xt[0, 0], cmap="gray", vmin=-1, vmax=1)
axes[1].set_title("Noisy xₜ (t=500)")
axes[2].imshow(x0_recovered[0, 0], cmap="gray", vmin=-1, vmax=1)
axes[2].set_title("Recovered x₀")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Load Trained Model and Sample

In [ ]:
import glob

checkpoints = sorted(glob.glob("artifacts/experiments/ddpm_mnist*/checkpoint/best.pt"))

if checkpoints:
    latest = checkpoints[-1]
    print(f"Loading: {latest}")
    model, schedule, payload = load_diffusion_checkpoint(latest)
    print(f"Step: {payload['step']}")
    print(f"Val loss: {payload.get('best_val_loss', 'N/A')}")
    print(f"Schedule: {schedule.kind}, T={schedule.timesteps}")
    print(f"Model params: {model.num_parameters():,}")
else:
    print("No checkpoint found. Train with: python scripts/train_diffusion.py")

## 5. Generate Samples

In [ ]:
if checkpoints:
    model.eval()

    # Generate 16 samples
    samples = sample_ddpm(
        model, (16, 1, 28, 28), schedule, device="cpu", generator=torch.Generator().manual_seed(42)
    )

    # Plot grid
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(samples[i, 0], cmap="gray", vmin=-1, vmax=1)
        ax.axis("off")

    plt.suptitle("DDPM Samples (from pure noise)")
    plt.tight_layout()
    plt.show()

## 6. Reverse Process Step-by-Step

In [ ]:
if checkpoints:
    from from_zero.diffusion import p_sample

    # Start from pure noise
    x = torch.randn(1, 1, 28, 28, generator=torch.Generator().manual_seed(999))

    # Sample timesteps to visualize
    vis_steps = [999, 750, 500, 250, 100, 50, 10, 5, 1, 0]

    fig, axes = plt.subplots(1, len(vis_steps), figsize=(20, 3))

    for ax, t in zip(axes, vis_steps):
        t_tensor = torch.tensor([t])
        x = p_sample(model, x, t_tensor, schedule)
        ax.imshow(x[0, 0], cmap="gray", vmin=-1, vmax=1)
        ax.set_title(f"t = {t}")
        ax.axis("off")

    plt.suptitle("Reverse Process: Noise → Image")
    plt.tight_layout()
    plt.show()

## 7. Schedule Identities Verification

In [ ]:
# Verify ᾱₜ = cumprod(αₛ)
schedule = make_schedule("linear", timesteps=100)
alpha_bars_manual = torch.cumprod(schedule.alphas, dim=0)
diff = torch.max(torch.abs(schedule.alpha_bars - alpha_bars_manual)).item()
print(f"ᾱₜ = cumprod(αₛ): max diff = {diff:.2e}")

# Verify posterior variance formula
prev_ab = schedule.previous_alpha_bars
expected_var = schedule.betas * (1 - prev_ab) / (1 - schedule.alpha_bars)
expected_var[0] = 0.0
diff_var = torch.max(torch.abs(schedule.posterior_variance - expected_var)).item()
print(f"Posterior variance formula: max diff = {diff_var:.2e}")

# Verify forward process moments
torch.manual_seed(0)
x0 = torch.full((10000, 1, 1, 1), 0.5)
t = 50
noise = torch.randn_like(x0)
xt = q_sample(x0, torch.full((10000,), t), noise, schedule)
expected_mean = torch.sqrt(schedule.alpha_bars[t]) * 0.5
expected_var = 1 - schedule.alpha_bars[t]
print(f"Forward mean: empirical={xt.mean():.4f}, expected={expected_mean:.4f}")
print(f"Forward var: empirical={xt.var():.4f}, expected={expected_var:.4f}")